In [44]:
!pip install fastapi uvicorn pydantic pandas scikit-learn joblib pytest

In [45]:
import pandas as pd
import joblib

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv(
    "rfm_modeling_snapshot.csv"
)

features = [

    "recency_days",
    "frequency_180d",
    "monetary_180d",
    "ticket_count_90d",
    "sessions_30d"

]

X = df[features]

y = df["churn_next_60d"]

model = Pipeline([

    (
        "imputer",
        SimpleImputer(
            strategy="median"
        )
    ),

    (
        "classifier",
        RandomForestClassifier(
            n_estimators=500,
            random_state=42
        )
    )

])

model.fit(X, y)

joblib.dump(
    model,
    "model.pkl"
)

print("Model saved successfully")

Model saved successfully


In [46]:
%%writefile app.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

import pandas as pd
import joblib

model = joblib.load(
    "model.pkl"
)

app = FastAPI(
    title="D2C Customer Churn API"
)

class CustomerRequest(BaseModel):

    recency_days: int = Field(..., ge=0)

    frequency_180d: int = Field(..., ge=0)

    monetary_180d: float = Field(..., ge=0)

    ticket_count_90d: int = Field(..., ge=0)

    sessions_30d: int = Field(..., ge=0)

class BatchRequest(BaseModel):

    customers: list[CustomerRequest]

def risk_reason(customer):

    reasons = []

    if customer.recency_days > 90:
        reasons.append("high inactivity")

    if customer.ticket_count_90d > 3:
        reasons.append("multiple complaints")

    if customer.sessions_30d < 3:
        reasons.append("low engagement")

    if customer.frequency_180d < 2:
        reasons.append("low purchase frequency")

    if not reasons:
        reasons.append("healthy customer behavior")

    return ", ".join(reasons)

@app.get("/health")
def health():

    return {

        "status": "ok"

    }

@app.post("/predict")
def predict(customer: CustomerRequest):

    try:

        data = pd.DataFrame([{

            "recency_days":
            customer.recency_days,

            "frequency_180d":
            customer.frequency_180d,

            "monetary_180d":
            customer.monetary_180d,

            "ticket_count_90d":
            customer.ticket_count_90d,

            "sessions_30d":
            customer.sessions_30d

        }])

        probability = float(
            model.predict_proba(data)[0][1]
        )

        prediction = int(
            probability >= 0.40
        )

        if probability >= 0.70:

            level = "high"

        elif probability >= 0.40:

            level = "medium"

        else:

            level = "low"

        return {

            "churn_probability":
            round(probability, 4),

            "predicted_class":
            prediction,

            "risk_level":
            level,

            "risk_explanation":
            risk_reason(customer)

        }

    except Exception as e:

        raise HTTPException(
            status_code=500,
            detail=str(e)
        )

@app.post("/batch_predict")
def batch_predict(
    request: BatchRequest
):

    predictions = []

    for customer in request.customers:

        predictions.append(
            predict(customer)
        )

    return {

        "predictions":
        predictions

    }

Overwriting app.py


In [47]:
import os

print(os.listdir())

['.config', 'test_batch_predict.py', 'model.pkl', '__pycache__', 'test_invalid_input.py', 'test_predict.py', 'rfm_modeling_snapshot.csv', '.pytest_cache', 'app.py', 'test_health.py', 'sample_data']


In [48]:
from fastapi.testclient import TestClient

from app import app

client = TestClient(app)

def test_health():

    response = client.get(
        "/health"
    )

    assert response.status_code == 200

    assert response.json()["status"] == "ok"

In [49]:
import app

print(app.app)

In [50]:
from fastapi.testclient import TestClient

from app import app

client = TestClient(app)

def test_health():

    response = client.get(
        "/health"
    )

    assert response.status_code == 200

    assert response.json()["status"] == "ok"

In [51]:
from fastapi.testclient import TestClient
from app import app

client = TestClient(app)

response = client.get("/health")

print(response.status_code)
print(response.json())

200
{'status': 'ok'}


In [52]:
from fastapi.testclient import TestClient

from app import app

client = TestClient(app)

def test_predict():

    payload = {

        "recency_days":120,
        "frequency_180d":1,
        "monetary_180d":900,
        "ticket_count_90d":5,
        "sessions_30d":1

    }

    response = client.post(
        "/predict",
        json=payload
    )

    assert response.status_code == 200

    data = response.json()

    assert "churn_probability" in data

    assert "predicted_class" in data

    assert "risk_level" in data

In [53]:
from fastapi.testclient import TestClient

from app import app

client = TestClient(app)

def test_batch():

    payload = {

        "customers":[

            {

                "recency_days":120,
                "frequency_180d":1,
                "monetary_180d":800,
                "ticket_count_90d":4,
                "sessions_30d":1

            },

            {

                "recency_days":10,
                "frequency_180d":8,
                "monetary_180d":9000,
                "ticket_count_90d":0,
                "sessions_30d":12

            }

        ]

    }

    response = client.post(
        "/batch_predict",
        json=payload
    )

    assert response.status_code == 200

    assert "predictions" in response.json()

In [54]:
from fastapi.testclient import TestClient

from app import app

client = TestClient(app)

def test_invalid_input():

    payload = {

        "recency_days":-10,
        "frequency_180d":1,
        "monetary_180d":500,
        "ticket_count_90d":1,
        "sessions_30d":2

    }

    response = client.post(
        "/predict",
        json=payload
    )

    assert response.status_code == 422

In [55]:
!pytest

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0
rootdir: /content
plugins: anyio-4.13.0, typeguard-4.5.2, langsmith-0.8.12
collected 4 items                                                              

test_batch_predict.py .                                                  [ 25%]
test_health.py .                                                         [ 50%]
test_invalid_input.py .                                                  [ 75%]
test_predict.py .                                                        [100%]

============================== 4 passed in 3.18s ===============================


In [56]:
import os

print(os.listdir())

['.config', 'test_batch_predict.py', 'model.pkl', '__pycache__', 'test_invalid_input.py', 'test_predict.py', 'rfm_modeling_snapshot.csv', '.pytest_cache', 'app.py', 'test_health.py', 'sample_data']


In [57]:
%%writefile test_health.py

from fastapi.testclient import TestClient
from app import app

client = TestClient(app)

def test_health():

    response = client.get("/health")

    assert response.status_code == 200

    assert response.json()["status"] == "ok"

Overwriting test_health.py


In [58]:
%%writefile test_predict.py

from fastapi.testclient import TestClient
from app import app

client = TestClient(app)

def test_predict():

    payload = {

        "recency_days":120,
        "frequency_180d":1,
        "monetary_180d":900,
        "ticket_count_90d":5,
        "sessions_30d":1

    }

    response = client.post(
        "/predict",
        json=payload
    )

    assert response.status_code == 200

    data = response.json()

    assert "churn_probability" in data

    assert "predicted_class" in data

    assert "risk_level" in data

Overwriting test_predict.py


In [59]:
%%writefile test_batch_predict.py

from fastapi.testclient import TestClient
from app import app

client = TestClient(app)

def test_batch():

    payload = {

        "customers":[

            {

                "recency_days":120,
                "frequency_180d":1,
                "monetary_180d":800,
                "ticket_count_90d":4,
                "sessions_30d":1

            },

            {

                "recency_days":10,
                "frequency_180d":8,
                "monetary_180d":9000,
                "ticket_count_90d":0,
                "sessions_30d":12

            }

        ]

    }

    response = client.post(
        "/batch_predict",
        json=payload
    )

    assert response.status_code == 200

    assert "predictions" in response.json()

Overwriting test_batch_predict.py


In [60]:
%%writefile test_invalid_input.py

from fastapi.testclient import TestClient
from app import app

client = TestClient(app)

def test_invalid_input():

    payload = {

        "recency_days":-10,
        "frequency_180d":1,
        "monetary_180d":500,
        "ticket_count_90d":1,
        "sessions_30d":2

    }

    response = client.post(
        "/predict",
        json=payload
    )

    assert response.status_code == 422

Overwriting test_invalid_input.py


In [61]:
!pytest --collect-only

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0
rootdir: /content
plugins: anyio-4.13.0, typeguard-4.5.2, langsmith-0.8.12
collected 4 items                                                              

<Dir content>
  <Module test_batch_predict.py>
    <Function test_batch>
  <Module test_health.py>
    <Function test_health>
  <Module test_invalid_input.py>
    <Function test_invalid_input>
  <Module test_predict.py>
    <Function test_predict>

========================== 4 tests collected in 5.93s ==========================


In [62]:
import os

print(os.listdir())

['.config', 'test_batch_predict.py', 'model.pkl', '__pycache__', 'test_invalid_input.py', 'test_predict.py', 'rfm_modeling_snapshot.csv', '.pytest_cache', 'app.py', 'test_health.py', 'sample_data']


In [63]:
from fastapi.testclient import TestClient
from app import app

client = TestClient(app)

response = client.get("/health")

print(response.json())

{'status': 'ok'}


In [64]:
response = client.post(
    "/predict",
    json={
        "recency_days":120,
        "frequency_180d":1,
        "monetary_180d":900,
        "ticket_count_90d":5,
        "sessions_30d":1
    }
)

print(response.json())

{'churn_probability': 0.728, 'predicted_class': 1, 'risk_level': 'high', 'risk_explanation': 'high inactivity, multiple complaints, low engagement, low purchase frequency'}


In [65]:
!pip install pyngrok

In [66]:
import threading
import uvicorn

def run():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )

threading.Thread(
    target=run
).start()

INFO:     Started server process [1917]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use


In [67]:
from pyngrok import ngrok

ngrok.set_auth_token("3CGZmBueb7jMhgUEvm190l9kHvU_815aniqoCJiRKBNzUJyiz")

INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


In [68]:
from pyngrok import ngrok

public_url = ngrok.connect(8000)

print(public_url)

NgrokTunnel: "https://jacket-upbeat-abreast.ngrok-free.dev" -> "http://localhost:8000"


In [69]:
from fastapi.testclient import TestClient
from app import app

client = TestClient(app)

print(client.get("/health").json())

{'status': 'ok'}


In [70]:
import threading
import uvicorn

from app import app

def run():

    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )

thread = threading.Thread(
    target=run,
    daemon=True
)

thread.start()

INFO:     Started server process [1917]
INFO:     Waiting for application startup.


In [71]:
import requests

response = requests.get(
    "http://127.0.0.1:8000/health"
)

print(response.json())

INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.


INFO:     127.0.0.1:44438 - "GET /health HTTP/1.1" 200 OK
{'status': 'ok'}


INFO:     Application shutdown complete.


In [72]:
import joblib

model = joblib.load("model.pkl")

print(type(model))

<class 'sklearn.pipeline.Pipeline'>


In [73]:
import app

print("Imported successfully")

Imported successfully


In [74]:
import requests

try:
    r = requests.get("http://127.0.0.1:8000/health")
    print(r.status_code)
    print(r.text)
except Exception as e:
    print(e)

INFO:     127.0.0.1:44440 - "GET /health HTTP/1.1" 200 OK
200
{"status":"ok"}


In [75]:
import uvicorn
import threading
from app import app

def run_uvicorn():
    config = uvicorn.Config(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="debug"
    )
    server = uvicorn.Server(config)
    server.run()

# Run uvicorn in a separate thread to avoid conflicts with the Colab event loop
thread = threading.Thread(target=run_uvicorn, daemon=True)
thread.start()

INFO:     Started server process [1917]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


In [76]:
import joblib
model = joblib.load("model.pkl")
print(type(model))

<class 'sklearn.pipeline.Pipeline'>


In [77]:
from pyngrok import ngrok

# Remove old tunnels
ngrok.kill()

# Create new tunnel
public_url = ngrok.connect(8000)

print(public_url)

NgrokTunnel: "https://jacket-upbeat-abreast.ngrok-free.dev" -> "http://localhost:8000"


In [78]:
from pyngrok import ngrok

for tunnel in ngrok.get_tunnels():
    print(tunnel.public_url)

https://jacket-upbeat-abreast.ngrok-free.dev
